# Аналитика League of Legends через Riot API

        ## Цель проекта

        Собрать данные League of Legends через Riot API и подготовить базовые таблицы для дальнейшей аналитики и дашборда.

        На первом этапе формируем три набора данных:

        - `players_df` — игроки верхних лиг;
        - `matches_df` — статистика участников матчей;
        - `champions_df` — справочник чемпионов.

        
**Что мы получим в итоге — 3 таблицы:**

| Файл | Что хранит |
|:--|:--|
| `players.csv` | список топовых игроков (Challenger) |
| `matches.csv` | матчи с подробной статистикой участников |
| `champions.csv` | справочник чемпионов |

        Этот ноутбук — чистая рабочая версия пайплайна. Черновики `lol_dataset.ipynb`, `lol_dataset_extended.ipynb` и `API_riot_reckon.ipynb` используем только как референсы.


## 1. Импорт библиотек

        Используем:

        - `requests` для запросов к Riot API;
        - `pandas` для таблиц;
        - `time` для пауз между запросами;
        - `getpass` для безопасного ввода API-ключа.


In [1]:
# Устанавливаем нужные библиотеки
!pip install -q requests pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\fiery\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import requests   # для HTTP-запросов к API
import pandas as pd  # для работы с таблицами
import time  # чтобы делать паузы между запросами и не получить бан от API
from getpass import getpass # вводим API-ключ скрыто, без отображения на экране

## 2. Настройка API и проверка ключа

        Development API Key Riot действует ограниченное время, обычно около 24 часов.

        Ключ не храним в ноутбуке: вставляем его при запуске ячейки.


In [ ]:
RIOT_API_KEY = getpass("Вставьте свежий RIOT_API_KEY:").strip()

# Для старта берем Европу. Регион матчей для EUW — europe.
REGION = "euw1"

MATCH_REGION = "europe"

headers = {
    "X-Riot-Token": RIOT_API_KEY
}

test_url = (
    f"https://{REGION}.api.riotgames.com"
    f"/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
)

response = requests.get(test_url, headers=headers, timeout=30)

print("Статус ответа:", response.status_code)

if response.status_code == 200:
    print("Ключ работает, можно продолжать.")
elif response.status_code in (401, 403):
    raise ValueError("Ключ недействителен или истек. Сгенерируйте новый ключ на developer.riotgames.com.")
elif response.status_code == 429:
    raise RuntimeError("Превышен лимит запросов Riot API. Подождите и повторите позже.")
else:
    raise RuntimeError(f"Неожиданный ответ API: {response.status_code}. {response.text[:300]}")


Статус ответа: 200
Ключ работает, можно продолжать.


## 3. Вспомогательная функция для запросов

        `safe_get()` делает запрос, проверяет статус ответа и добавляет паузу, чтобы не упереться в rate limits.


In [8]:
def safe_get(url, headers=headers, pause=1.2):
            response = requests.get(url, headers=headers, timeout=30)

            if response.status_code == 200:
                time.sleep(pause)
                return response.json()

            if response.status_code in (401, 403):
                raise ValueError(f"Проблема с API-ключом: статус {response.status_code}")

            if response.status_code == 429:
                raise RuntimeError("Превышен лимит запросов Riot API.")

            print(f"Запрос не выполнен: {response.status_code}")
            print(url)
            print(response.text[:300])
            return None


## 4. Получение игроков Challenger

        Начинаем с Challenger-лиги: это стабильная точка входа, потому что в ответе есть `puuid` игроков.

        `puuid` нужен дальше для Match API.


In [9]:
league_url = (
            f"https://{REGION}.api.riotgames.com"
            f"/lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
        )

league_data = safe_get(league_url)

players_df = pd.DataFrame(league_data["entries"])
players_df["region"] = REGION
players_df["tier"] = league_data["tier"]
players_df["queue"] = league_data["queue"]
players_df["winrate"] = players_df["wins"] / (players_df["wins"] + players_df["losses"])

players_df = players_df.sort_values("leaguePoints", ascending=False).reset_index(drop=True)

players_df.head()


,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak,region,tier,queue,winrate
0,kUdJh-6XcgbS1PnyMw8P0OkfxgDWEa26wDlQA8N9CKIB8N...,3530,I,701,556,True,False,False,False,euw1,CHALLENGER,RANKED_SOLO_5x5,0.557677
1,I54lhHo4qTV_q_Gf5OSXsHvZGqyhxZ8Wl71ze3nBcvopND...,3289,I,581,411,True,False,False,True,euw1,CHALLENGER,RANKED_SOLO_5x5,0.585685
2,gordHOvwe2mXFmEKm98xrQJd2pGPrDzA-KSs3vtkefiSym...,3074,I,438,368,True,False,False,False,euw1,CHALLENGER,RANKED_SOLO_5x5,0.543424
3,fLsqOC1hTfy2_i9t3JGx48mU-9YYxswi47luCFD7xmZhQp...,3047,I,303,239,True,False,False,False,euw1,CHALLENGER,RANKED_SOLO_5x5,0.559041
4,NKjterzII-UiiQaMw7xjHmMVt4UbRI8apzIkIWhgIm0zSI...,3038,I,372,242,True,False,False,False,euw1,CHALLENGER,RANKED_SOLO_5x5,0.605863


## 5. Описание таблицы `players_df`

        - `puuid` — уникальный идентификатор игрока, нужен для получения матчей.
        - `leaguePoints` — очки лиги.
        - `rank` — ранг внутри текущего tier.
        - `wins`, `losses` — победы и поражения в лиге.
        - `veteran`, `inactive`, `freshBlood`, `hotStreak` — служебные статусы Riot.
        - `region` — сервер игрока.
        - `tier` — уровень лиги.
        - `queue` — тип очереди.
        - `winrate` — доля побед игрока.


In [10]:
print("Размер players_df:", players_df.shape)
players_df.info()

Размер players_df: (323, 13)
<class 'pandas.DataFrame'>
RangeIndex: 323 entries, 0 to 322
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   puuid         323 non-null    str    
 1   leaguePoints  323 non-null    int64  
 2   rank          323 non-null    str    
 3   wins          323 non-null    int64  
 4   losses        323 non-null    int64  
 5   veteran       323 non-null    bool   
 6   inactive      323 non-null    bool   
 7   freshBlood    323 non-null    bool   
 8   hotStreak     323 non-null    bool   
 9   region        323 non-null    str    
 10  tier          323 non-null    str    
 11  queue         323 non-null    str    
 12  winrate       323 non-null    float64
dtypes: bool(4), float64(1), int64(3), str(5)
memory usage: 24.1 KB


## 6. Получение match_id для игроков

        Для первых игроков из `players_df` получаем список матчей.

        Пока берем небольшой объем, чтобы не тратить лимиты API.


In [11]:
TOP_PLAYERS = 10
MATCHES_PER_PLAYER = 5

match_ids = []

for puuid in players_df.head(TOP_PLAYERS)["puuid"]:
    ids_url = (
        f"https://{MATCH_REGION}.api.riotgames.com"
        f"/lol/match/v5/matches/by-puuid/{puuid}/ids"
        f"?queue=420&start=0&count={MATCHES_PER_PLAYER}"
    )

    data = safe_get(ids_url)

    if data:
        match_ids.extend(data)

match_ids = sorted(set(match_ids))

print("Количество уникальных матчей:", len(match_ids))
match_ids[:5]


Количество уникальных матчей: 42


['EUW1_7871067712',
 'EUW1_7871111360',
 'EUW1_7872107370',
 'EUW1_7872191598',
 'EUW1_7872272444']

## 7. Получение деталей матчей

        По каждому `match_id` получаем подробности матча и извлекаем статистику каждого участника.


In [12]:
match_rows = []

for match_id in match_ids:
    match_url = (
        f"https://{MATCH_REGION}.api.riotgames.com"
        f"/lol/match/v5/matches/{match_id}"
    )

    match_data = safe_get(match_url)

    if not match_data:
        continue

    info = match_data["info"]

    for participant in info["participants"]:
        row = {
            "match_id": match_id,
            "region": REGION,
            "game_creation": info.get("gameCreation"),
            "game_duration_sec": info.get("gameDuration"),
            "game_mode": info.get("gameMode"),
            "game_version": info.get("gameVersion"),
            "puuid": participant.get("puuid"),
            "summoner_name": participant.get("summonerName"),
            "champion_id": participant.get("championId"),
            "champion": participant.get("championName"),
            "team_id": participant.get("teamId"),
            "team_position": participant.get("teamPosition"),
            "kills": participant.get("kills"),
            "deaths": participant.get("deaths"),
            "assists": participant.get("assists"),
            "win": participant.get("win"),
            "gold_earned": participant.get("goldEarned"),
            "damage_to_champions": participant.get("totalDamageDealtToChampions"),
            "minions_killed": participant.get("totalMinionsKilled"),
            "vision_score": participant.get("visionScore"),
            "item0": participant.get("item0"),
            "item1": participant.get("item1"),
            "item2": participant.get("item2"),
            "item3": participant.get("item3"),
            "item4": participant.get("item4"),
            "item5": participant.get("item5"),
            "item6": participant.get("item6"),
        }
        match_rows.append(row)

matches_df = pd.DataFrame(match_rows)

print("Размер matches_df:", matches_df.shape)
matches_df.head()


Размер matches_df: (420, 27)


,match_id,region,game_creation,game_duration_sec,game_mode,game_version,puuid,summoner_name,champion_id,champion,...,damage_to_champions,minions_killed,vision_score,item0,item1,item2,item3,item4,item5,item6
0,EUW1_7871067712,euw1,1780168616060,1538,CLASSIC,16.11.781.1789,7iA0Izb_sWhigYiqUUgJiPtl1I3sLMdjcOgEfv1Z7Msoc2...,,67,Vayne,...,19991,221,26,1042,3124,1086,3047,3073,3044,3363
1,EUW1_7871067712,euw1,1780168616060,1538,CLASSIC,16.11.781.1789,SffV0aXN5LDHYg6Mph-ScGvK9UeheuwZCko_KcX0G-XPHC...,,72,Skarner,...,18334,11,24,2525,3143,3047,2524,1011,1011,3364
2,EUW1_7871067712,euw1,1780168616060,1538,CLASSIC,16.11.781.1789,ImObDyLCeYjv96CflJ-lD5_7FwqNIkImXLOKaRGJ9nuy-x...,,34,Anivia,...,15663,182,16,1082,3145,3108,3118,3170,6657,3363
3,EUW1_7871067712,euw1,1780168616060,1538,CLASSIC,16.11.781.1789,31Eyfdagy5c1BInmgUnkCCBiJv6Xjx30_XQnMGUdIgyixY...,,96,KogMaw,...,34625,239,24,1086,3082,3085,3153,3302,3124,3363
4,EUW1_7871067712,euw1,1780168616060,1538,CLASSIC,16.11.781.1789,-XJd1pZ7mihH683LLRnIIwAJm7QY9nfWhD_jzRstMDEClw...,,44,Taric,...,6677,29,58,3109,3870,3190,3047,3070,1028,3364


## 8. Описание таблицы `matches_df`

        - `match_id` — идентификатор матча.
        - `region` — регион, из которого получен матч.
        - `game_creation` — время создания матча в Unix milliseconds.
        - `game_duration_sec` — длительность матча в секундах.
        - `game_mode` — режим игры.
        - `game_version` — версия игры.
        - `puuid` — идентификатор участника.
        - `summoner_name` — имя игрока, если Riot отдал его в ответе.
        - `champion_id`, `champion` — чемпион.
        - `team_id` — команда: 100 или 200.
        - `team_position` — позиция: TOP, JUNGLE, MIDDLE, BOTTOM, UTILITY.
        - `kills`, `deaths`, `assists` — базовые боевые показатели.
        - `win` — победил ли участник.
        - `gold_earned`, `damage_to_champions`, `minions_killed`, `vision_score` — игровые метрики.
        - `item0` ... `item6` — предметы игрока.


In [13]:
matches_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   match_id             420 non-null    str  
 1   region               420 non-null    str  
 2   game_creation        420 non-null    int64
 3   game_duration_sec    420 non-null    int64
 4   game_mode            420 non-null    str  
 5   game_version         420 non-null    str  
 6   puuid                420 non-null    str  
 7   summoner_name        420 non-null    str  
 8   champion_id          420 non-null    int64
 9   champion             420 non-null    str  
 10  team_id              420 non-null    int64
 11  team_position        420 non-null    str  
 12  kills                420 non-null    int64
 13  deaths               420 non-null    int64
 14  assists              420 non-null    int64
 15  win                  420 non-null    bool 
 16  gold_earned          420 non-null    

## 9. Добавление производных метрик

        Добавляем показатели, которые пригодятся для анализа:

        - `kda`;
        - длительность матча в минутах;
        - урон в минуту;
        - золото в минуту.


In [14]:
matches_df["kda"] = ((matches_df["kills"] + matches_df["assists"])/ matches_df["deaths"].clip(lower=1)        )
matches_df["game_duration_min"] = matches_df["game_duration_sec"] / 60
matches_df["damage_per_min"] = matches_df["damage_to_champions"] / matches_df["game_duration_min"]
matches_df["gold_per_min"] = matches_df["gold_earned"] / matches_df["game_duration_min"]

matches_df[[
    "champion", "team_position", "kills", "deaths", "assists",
    "kda", "game_duration_min", "damage_per_min", "gold_per_min"
]].head(10)


,champion,team_position,kills,deaths,assists,kda,game_duration_min,damage_per_min,gold_per_min
0,Vayne,TOP,4,4,3,1.750000,25.633333,779.882965,424.291287
1,Skarner,JUNGLE,5,3,15,6.666667,25.633333,715.240572,448.517555
2,Anivia,MIDDLE,4,4,11,3.750000,25.633333,611.040312,395.851756
3,KogMaw,BOTTOM,15,1,5,20.000000,25.633333,1350.780234,645.643693
4,Taric,UTILITY,1,2,20,10.500000,25.633333,260.481144,301.521456
5,Ambessa,TOP,2,7,3,0.714286,25.633333,565.474642,329.414824
6,Ekko,JUNGLE,4,7,5,1.285714,25.633333,947.867360,423.003901
7,Akshan,MIDDLE,6,6,4,1.666667,25.633333,907.568270,477.152146
8,Jhin,BOTTOM,2,2,2,2.000000,25.633333,453.667100,400.299090
9,Rakan,UTILITY,0,7,7,1.000000,25.633333,177.815345,240.351105


## 10. Получение справочника чемпионов

        Data Dragon — статический источник Riot. Для него не нужен API-ключ.


In [15]:
versions_url = "https://ddragon.leagueoflegends.com/api/versions.json"
latest_version = requests.get(versions_url, timeout=30).json()[0]

champions_url = (
    f"https://ddragon.leagueoflegends.com/cdn/"
    f"{latest_version}/data/en_US/champion.json"
)

champions_data = requests.get(champions_url, timeout=30).json()["data"]

champion_rows = []

for champion_name, champion_info in champions_data.items():
    champion_rows.append({
        "champion_id": int(champion_info["key"]),
        "champion_name": champion_info["name"],
        "title": champion_info["title"],
        "tags": ",".join(champion_info["tags"]),
    })

champions_df = pd.DataFrame(champion_rows).sort_values("champion_name").reset_index(drop=True)

print("Версия Data Dragon:", latest_version)
print("Размер champions_df:", champions_df.shape)
champions_df.head()


Версия Data Dragon: 16.11.1
Размер champions_df: (172, 4)


,champion_id,champion_name,title,tags
0,266,Aatrox,the Darkin Blade,Fighter
1,103,Ahri,the Nine-Tailed Fox,"Mage,Assassin"
2,84,Akali,the Rogue Assassin,Assassin
3,166,Akshan,the Rogue Sentinel,"Marksman,Assassin"
4,12,Alistar,the Minotaur,"Tank,Support"


## 11. Первичная проверка данных

        Проверяем размеры таблиц, пропуски и базовые статистики.


In [16]:
print("players_df:", players_df.shape)
print("matches_df:", matches_df.shape)
print("champions_df:", champions_df.shape)


players_df: (323, 13)
matches_df: (420, 31)
champions_df: (172, 4)


In [17]:
matches_df[[
            "kda",
            "game_duration_min",
            "damage_per_min",
            "gold_per_min"
        ]].describe()


,kda,game_duration_min,damage_per_min,gold_per_min
count,420.000000,420.000000,420.000000,420.000000
mean,4.095113,26.342460,706.633706,421.782401
std,4.605895,5.427709,325.613196,96.629928
min,0.000000,15.350000,155.779037,236.260623
25%,1.333333,23.166667,468.080100,353.518648
50%,2.500000,26.050000,668.294839,416.617525
75%,5.000000,31.050000,907.637939,479.493052
max,28.000000,38.533333,2445.340720,754.605809


## 12. Первый аналитический агрегат по чемпионам

        Считаем количество игр, win rate и средние игровые метрики по чемпионам.


In [18]:
champion_stats = (
            matches_df
            .groupby("champion")
            .agg(
                games=("match_id", "count"),
                winrate=("win", "mean"),
                avg_kda=("kda", "mean"),
                avg_damage_per_min=("damage_per_min", "mean"),
                avg_gold_per_min=("gold_per_min", "mean"),
            )
            .reset_index()
        )

champion_stats["winrate"] = champion_stats["winrate"] * 100

champion_stats.sort_values("games", ascending=False).head(15)


,champion,games,winrate,avg_kda,avg_damage_per_min,avg_gold_per_min
1,Ahri,15,40.000000,3.617063,705.526817,369.167223
38,Graves,9,44.444444,3.987302,756.808757,497.305616
96,Senna,9,55.555556,5.402160,911.583775,436.908806
46,Jayce,8,50.000000,2.144345,751.585818,409.260613
20,Cassiopeia,8,50.000000,3.358333,993.799033,429.627138
117,Varus,8,62.500000,6.644034,979.776144,475.636245
130,Zeri,7,71.428571,4.710884,1001.328085,602.058935
105,Skarner,7,28.571429,4.876701,549.777284,355.385202
40,Hecarim,7,42.857143,3.543424,819.246349,466.545242
51,Kalista,7,42.857143,2.240079,980.816051,529.250725


## 13. Сохранение данных

        Сохраняем текущую версию таблиц в CSV. Эти файлы можно использовать дальше для анализа и дашборда.


In [19]:
players_df.to_csv("players_data.csv", index=False)
matches_df.to_csv("matches_data.csv", index=False)
champions_df.to_csv("champions_data.csv", index=False)
champion_stats.to_csv("champion_stats.csv", index=False)

print("Сохранены файлы:")
print("- players_data.csv")
print("- matches_data.csv")
print("- champions_data.csv")
print("- champion_stats.csv")


Сохранены файлы:
- players_data.csv
- matches_data.csv
- champions_data.csv
- champion_stats.csv


## 14. Итоги API-этапа

На этом этапе был собран демонстрационный датасет через Riot API.

Что получилось:

- получен список игроков Challenger-лиги региона EUW;
- из данных игроков взяты `puuid`, которые нужны для Match API;
- по `puuid` получены идентификаторы матчей;
- по `match_id` загружены детали матчей;
- из JSON-ответов собрана таблица участников матчей;
- через Data Dragon получен справочник чемпионов;
- рассчитаны первые производные метрики: `kda`, `game_duration_min`, `damage_per_min`, `gold_per_min`;
- сохранены CSV-файлы для дальнейшей работы.

API-пайплайн показывает полный ETL-цикл:

| Шаг | Что сделано |
|:--|:--|
| Extract | Получили данные из Riot API и Data Dragon |
| Transform | Разобрали JSON и превратили его в таблицы |
| Load | Сохранили результаты в CSV |

Ограничения текущего API-этапа:

- собран только один регион: `euw1`;
- собрана только Challenger-лига;
- объем матчей небольшой, потому что Development API Key имеет лимиты;
- ключ Riot действует ограниченное время;
- для большого анализа через API нужно вести лог скачанных матчей и не загружать одни и те же `match_id` повторно.

Вывод:

API-часть оставляем как демонстрацию навыка работы с внешним API и ETL. Для полноценного анализа и более надежных выводов дальше используем большой Kaggle-датасет.



## Что сделать дальше

        - Увеличить объем данных: больше игроков, больше матчей.
        - Добавить второй регион `na1`.
        - Сравнить регионы Европа / США.
        - Построить графики для дашборда.
        - Перенести финальный пайплайн в `data_pipeline.py` или оставить в чистом ноутбуке.


## 15. Инкрементальный сбор данных через Riot API

После встречи с наставником решили не пытаться сразу скачать весь Riot API. Вместо этого добавляем отдельный сборщик, который можно запускать маленькими порциями и постепенно накапливать собственный датасет.

Логика сбора:

1. Берем игроков из верхних лиг: `challenger`, `grandmaster`, `master`.
2. Для каждого игрока получаем список `match_id` за выбранный период или последние матчи.
3. По каждому `match_id` скачиваем детали матча.
4. Сохраняем всех 10 участников матча.
5. Ведем `download_log_api.csv`, чтобы не скачивать один матч повторно.

Основной анализ пока делаем на Kaggle-датасете, а API-сборщик используем как сильную ETL-часть проекта и источник свежих данных.


## 16. Параметры запуска сборщика

Актуальная версия использует `PLAYER_OFFSET`: это смещение по списку игроков лиги. Если при повторном запуске данные не растут, увеличиваем `PLAYER_OFFSET`, чтобы брать следующих игроков, а не тех же самых.


In [ ]:
# АКТУАЛЬНАЯ ячейка параметров: есть PLAYER_OFFSET
COLLECTOR_TIER = "master"
PLAYER_OFFSET = 20
MAX_PLAYERS = 10
MATCHES_PER_PLAYER = 5

print("Запуск сборщика:")
print("Лига:", COLLECTOR_TIER)
print("Смещение по списку игроков:", PLAYER_OFFSET)
print("Игроков:", MAX_PLAYERS)
print("Матчей на игрока:", MATCHES_PER_PLAYER)
print("Ожидаемый максимум строк участников:", MAX_PLAYERS * MATCHES_PER_PLAYER * 10)


## 17. Запуск сборщика

При запуске ячейка попросит свежий `RIOT_API_KEY`. Ключ не сохраняется в ноутбук, поэтому его безопаснее вставлять вручную.

Если ключ истек, нужно получить новый на Riot Developer Portal и запустить ячейку еще раз.


In [ ]:
# АКТУАЛЬНАЯ ячейка запуска: параметры передаются через sys.argv.
# Скрипт лежит в папке scripts, а данные сохраняет в data/api.

import runpy
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "scripts").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPT_PATH = PROJECT_ROOT / "scripts" / "riot_data_collector.py"

sys.argv = [
    str(SCRIPT_PATH),
    "--tier", COLLECTOR_TIER,
    "--player-offset", str(PLAYER_OFFSET),
    "--max-players", str(MAX_PLAYERS),
    "--matches-per-player", str(MATCHES_PER_PLAYER),
]

runpy.run_path(str(SCRIPT_PATH), run_name="__main__")


## 18. Проверка накопленных API-данных

После запуска проверяем, какие файлы появились и сколько строк уже накопилось.


In [ ]:
from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent


api_dir = PROJECT_ROOT / "data" / "api"
api_files = [
    "players_api.csv",
    "matches_api.csv",
    "download_log_api.csv",
]

for file_name in api_files:
    file_path = api_dir / file_name
    if file_path.exists():
        df_check = pd.read_csv(file_path)
        print(f"{file_name}: {df_check.shape[0]} строк, {df_check.shape[1]} колонок")
    else:
        print(f"{file_name}: файл пока не создан")


In [ ]:
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent


matches_api_df = pd.read_csv(PROJECT_ROOT / "data" / "api" / "matches_api.csv")
log_api_df = pd.read_csv(PROJECT_ROOT / "data" / "api" / "download_log_api.csv")

print("Строк участников:", len(matches_api_df))
print("Уникальных матчей:", matches_api_df["match_id"].nunique())
print("Уникальных игроков:", matches_api_df["puuid"].nunique())
print("Чемпионов:", matches_api_df["champion_name"].nunique())

matches_api_df.head()


In [8]:
matches_api_df.groupby("source_tier")["match_id"].nunique()

source_tier
challenger     120
grandmaster    123
master         114
Name: match_id, dtype: int64

In [ ]:
# Добавляем производные метрики для анализа качества игры.
# deaths.clip(lower=1) защищает от деления на ноль, если игрок ни разу не умер.
matches_api_df["kda"] = (matches_api_df["kills"] + matches_api_df["assists"]) / matches_api_df["deaths"].clip(lower=1)
matches_api_df["game_duration_min"] = matches_api_df["game_duration_sec"] / 60
matches_api_df["damage_per_min"] = matches_api_df["total_damage_dealt_to_champions"] / matches_api_df["game_duration_min"]
matches_api_df["gold_per_min"] = matches_api_df["gold_earned"] / matches_api_df["game_duration_min"]
matches_api_df["vision_per_min"] = matches_api_df["vision_score"] / matches_api_df["game_duration_min"]


In [ ]:
# Первый аналитический срез: сравниваем верхние лиги по объему и средним игровым метрикам.
# winrate здесь равен примерно 0.5, потому что в каждом матче 5 победителей и 5 проигравших.
api_tier_stats = (
    matches_api_df
    .groupby("source_tier")
    .agg(
        matches=("match_id", "nunique"),
        players=("puuid", "nunique"),
        avg_kda=("kda", "mean"),
        avg_damage_per_min=("damage_per_min", "mean"),
        avg_gold_per_min=("gold_per_min", "mean"),
        winrate=("win", "mean"),
    )
    .reset_index()
)

api_tier_stats


In [ ]:
# Контроль качества: каждый матч должен быть представлен 10 строками участников.
# Если результат 10 -> количество матчей, структура данных собрана корректно.
matches_api_df["match_id"].value_counts().value_counts()


## 19. Как масштабируем сбор

Если пробный запуск прошел нормально, увеличиваем объем постепенно:

- сначала `--max-players 10 --matches-per-player 5`;
- потом `--max-players 30 --matches-per-player 10`;
- затем можно отдельно запустить `grandmaster` и `master`.

Цель для собственной API-выборки: накопить хотя бы 1000-3000 строк участников. Для финального дашборда основной объем можно брать из Kaggle, а API-выборку использовать как демонстрацию свежего сбора данных.


## 20. Итоги собственного API-сбора

Через Riot API была собрана собственная выборка матчей по верхним лигам `challenger`, `grandmaster` и `master`.

Итоговый объем API-выборки:

- 357 уникальных матчей;
- 3570 строк участников;
- 1707 уникальных игроков;
- 171 чемпион;
- по каждому матчу сохранены все 10 участников.

Распределение матчей по исходной лиге:

- Challenger — 120 матчей;
- Grandmaster — 123 матча;
- Master — 114 матчей.

Контроль качества показал, что каждый матч представлен ровно 10 строками участников. Это подтверждает корректность структуры данных: одна строка соответствует одному участнику матча.

API-выборка будет использоваться как демонстрация собственного ETL-пайплайна и как дополнительный источник для первичного анализа. Основной дашборд можно строить на более крупном Kaggle-датасете, а API-данные использовать для сравнения и демонстрации свежего сбора.


In [ ]:
# Сохраняем API-таблицу с производными метриками для дальнейшего анализа.
# Этот файл уже удобнее использовать в EDA: основные расчетные поля добавлены заранее.


PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent


output_path = PROJECT_ROOT / "data" / "api" / "matches_api_enriched.csv"
matches_api_df.to_csv(output_path, index=False)

print("Сохранено:", output_path)
print("Размер таблицы:", matches_api_df.shape)


## 21. Что использовали на API-этапе

На API-этапе использовались два основных файла:

- `LOL_project_pipeline.ipynb` — рабочий ноутбук, где задаются параметры запуска, запускается сборщик, проверяется качество данных и считаются первые метрики.
- `riot_data_collector.py` — отдельный Python-скрипт сборщика, который обращается к Riot API, сохраняет игроков, матчи и лог скачивания.

Результаты API-сбора лежат в папке `data/api`:

- `players_api.csv` — стартовые игроки из выбранных верхних лиг;
- `matches_api.csv` — сырые строки участников матчей;
- `matches_api_enriched.csv` — строки участников с расчетными метриками;
- `download_log_api.csv` — лог скачанных матчей, который защищает от повторной загрузки дублей.

Ключевая логика пайплайна: верхняя лига -> игроки -> PUUID -> match_id -> детали матча -> 10 строк участников -> CSV/Parquet -> проверка качества -> расчет метрик.
